# Module 10 -- Risk Metrics

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

VaR is the number your risk department asks for. It tells you
almost nothing about the risk that matters. But regulators require
it, and understanding its flaws teaches you more about risk than
the metric itself. I have watched VaR sit calmly while the real
risk migrated to correlation breakdowns and liquidity gaps.
Then the loss happens and VaR catches up the next day. Too late.

---
*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


## Synthetic Portfolio Returns

10 years of daily returns for a 3-asset portfolio (equity, bonds, gold)
with fat tails and injected historical crash scenarios.


In [ ]:
np.random.seed(2024)
n_days = 2520  # ~10 years of trading days

# Correlated asset returns using Cholesky decomposition
# Assets: equity index, bonds, gold
rho_matrix = np.array([
    [1.00,  -0.30,  0.05],
    [-0.30,  1.00,  0.15],
    [0.05,   0.15,  1.00]
])
L = np.linalg.cholesky(rho_matrix)

# Daily parameters (annualized: equity ~8% return, 18% vol)
daily_mu = np.array([0.08, 0.03, 0.04]) / 252
daily_sigma = np.array([0.18, 0.06, 0.15]) / np.sqrt(252)

# Generate correlated normals, then add fat tails via t-distribution mix
z = np.random.standard_normal((n_days, 3))
# 5% of days are "stress" days with 3x vol (simple fat tail injection)
stress_mask = np.random.random(n_days) < 0.05
z[stress_mask] *= 3.0

correlated_z = z @ L.T
returns = daily_mu + daily_sigma * correlated_z

# Portfolio: 60% equity, 30% bonds, 10% gold
weights = np.array([0.60, 0.30, 0.10])
portfolio_returns = returns @ weights

# Inject historical-style crashes
# 2008-style: 5 consecutive days of -3% to -7%
crash_start = 500
portfolio_returns[crash_start:crash_start+5] = [-0.03, -0.05, -0.04, -0.07, -0.02]

# COVID March 2020-style: 3 days of extreme moves
covid_start = 1800
portfolio_returns[covid_start:covid_start+3] = [-0.06, 0.04, -0.08]

# VIX Feb 2018 (Volmageddon): single-day spike
volmageddon = 1200
portfolio_returns[volmageddon] = -0.045

print(f"Portfolio: {n_days} days, ann. return {portfolio_returns.mean()*252:.2%}, "
      f"ann. vol {portfolio_returns.std()*np.sqrt(252):.2%}, "
      f"worst day {portfolio_returns.min():.2%}")


## Historical VaR

Sort the returns, pick the percentile. No model risk. Only weakness:
it only knows about the past.


In [ ]:
confidence_95 = 0.95
confidence_99 = 0.99

# Historical VaR -- just the empirical quantile
var_95_hist = -np.percentile(portfolio_returns, (1 - confidence_95) * 100)
var_99_hist = -np.percentile(portfolio_returns, (1 - confidence_99) * 100)

# Expected Shortfall (CVaR) -- average of losses beyond VaR
losses_beyond_95 = portfolio_returns[portfolio_returns < -var_95_hist]
losses_beyond_99 = portfolio_returns[portfolio_returns < -var_99_hist]
es_95 = -losses_beyond_95.mean()
es_99 = -losses_beyond_99.mean()

print(f"VaR 95%: {var_95_hist:.4%} | VaR 99%: {var_99_hist:.4%}")
print(f"ES  95%: {es_95:.4%} | ES  99%: {es_99:.4%}")
print(f"On $10M -- VaR99: ${var_99_hist * 10e6:,.0f}, ES99: ${es_99 * 10e6:,.0f}")


## Parametric VaR

Assumes normal returns. Fast, easy to decompose. Wrong in the tails.


In [ ]:
# Parametric VaR using portfolio mean and standard deviation
port_mu = portfolio_returns.mean()
port_sigma = portfolio_returns.std()

var_95_param = -(port_mu + norm.ppf(1 - confidence_95) * port_sigma)
var_99_param = -(port_mu + norm.ppf(1 - confidence_99) * port_sigma)

print(f"Parametric VaR -- 95%: {var_95_param:.4%}, 99%: {var_99_param:.4%}")
print(f"Historical VaR -- 95%: {var_95_hist:.4%}, 99%: {var_99_hist:.4%}")
print(f"Parametric underestimates 99% by: {(var_99_hist - var_99_param) / var_99_hist:.1%}")


## P&L Distribution with VaR Lines


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(portfolio_returns * 100, bins=100, color='#2196F3', alpha=0.7, edgecolor='none', density=True)
ax.axvline(-var_95_hist * 100, color='orange', linewidth=2, linestyle='--', label=f'VaR 95%')
ax.axvline(-var_99_hist * 100, color='red', linewidth=2, linestyle='--', label=f'VaR 99%')
x_range = np.linspace(portfolio_returns.min() * 100, portfolio_returns.max() * 100, 200)
ax.plot(x_range, norm.pdf(x_range, port_mu * 100, port_sigma * 100),
        'k-', linewidth=1.5, alpha=0.5, label='Normal fit')
ax.set_xlabel('Daily Return (%)'); ax.set_ylabel('Density')
ax.set_title('Portfolio P&L Distribution with VaR')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../data/10_pnl_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


The left tail is fatter than the normal -- that gap is where VaR breaks.

## Stress Testing

The worst days in history applied to our portfolio. More useful than VaR.


In [ ]:
# Historical stress scenarios -- approximate daily equity impact
# These are stylized for the 60/30/10 portfolio
scenarios = {
    'Lehman (Sep 2008)':     [-0.048, 0.015, 0.02],
    'Oct 2008 worst day':    [-0.09,  0.01, -0.03],
    'COVID (Mar 16, 2020)':  [-0.12, -0.02, -0.04],
    'Volmageddon (Feb 2018)':[-0.042,-0.005,-0.01],
    'Flash Crash (May 2010)':[-0.034, 0.01,  0.005],
}

print(f"{'Scenario':<28} {'Equity':>8} {'Bonds':>8} {'Gold':>8} {'Portfolio':>10}")
print('-' * 65)
for name, rets in scenarios.items():
    port = weights @ np.array(rets)
    print(f"{name:<28} {rets[0]:>7.1%} {rets[1]:>7.1%} {rets[2]:>7.1%} {port:>9.2%}")


COVID: everything dropped together. Your correlation matrix is a
fair-weather friend.

## Correlation Matrix


In [ ]:
asset_names = ['Equity', 'Bonds', 'Gold']
corr = np.corrcoef(returns.T)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                fontsize=14, fontweight='bold', color='white' if abs(corr[i,j]) > 0.5 else 'black')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(asset_names); ax.set_yticklabels(asset_names)
ax.set_title('Realized Correlation Matrix')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('../data/10_correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()


## Rolling VaR and Backtest

VaR spikes *after* the crash. It tells you yesterday's risk today.


In [ ]:
window = 252
rolling_var_95 = np.full(n_days, np.nan)
rolling_var_99 = np.full(n_days, np.nan)

for i in range(window, n_days):
    w = portfolio_returns[i - window:i]
    rolling_var_95[i] = -np.percentile(w, 5)
    rolling_var_99[i] = -np.percentile(w, 1)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
cum_returns = np.cumprod(1 + portfolio_returns) - 1
axes[0].plot(cum_returns * 100, 'b-', linewidth=0.8)
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].set_title('Portfolio Performance and Rolling VaR')
axes[1].plot(rolling_var_95 * 100, 'orange', linewidth=1.2, label='VaR 95%')
axes[1].plot(rolling_var_99 * 100, 'red', linewidth=1.2, label='VaR 99%')
axes[1].set_xlabel('Trading Day'); axes[1].set_ylabel('VaR (%)'); axes[1].legend()
plt.tight_layout()
plt.savefig('../data/10_rolling_var.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Backtest -- count breaches
breaches_95 = sum(1 for i in range(window, n_days) if portfolio_returns[i] < -rolling_var_95[i])
breaches_99 = sum(1 for i in range(window, n_days) if portfolio_returns[i] < -rolling_var_99[i])
total_valid = n_days - window
print(f"VaR backtest ({total_valid} days):")
print(f"  95% breaches: {breaches_95} ({breaches_95/total_valid:.1%}) -- expected ~5%")
print(f"  99% breaches: {breaches_99} ({breaches_99/total_valid:.1%}) -- expected ~1%")


## The Real Talk

After 17 years on a desk: VaR is a regulatory checkbox, not a
risk management tool. The tail is where the money is lost and VaR
tells you nothing about the tail. Correlation goes to 1 in a crisis.
The useful metrics -- liquidity-adjusted VaR, basis risk, gamma of
gamma -- are the ones nobody reports. Stress testing beats VaR
every time. Pick five worst scenarios and compute the loss.

---

## Books & Papers by the Author

For deeper treatment of the topics covered in this toolkit:

- [Beyond Gamma Exposure: A Four-Lens Framework for Dealer Positioning](https://www.amazon.com/dp/B0H2RZGMY6)
  -- GEX, vanna, charm, volga ([working paper](https://doi.org/10.5281/zenodo.20509786))

- [FX Traders vs Brokers: Vanilla and Exotic Options, Forwards, and Other OTC Structures](https://www.amazon.com/dp/B0H3VSV88X)
  -- Exotics, barriers, FX skew ([working paper](https://doi.org/10.5281/zenodo.20509708))

- [The China AI Disruption Thesis](https://www.amazon.com/dp/B0H11WH3R9)
  -- AI infrastructure repricing ([working paper](https://doi.org/10.5281/zenodo.20509816))

- [The Coming Shadow Banking Crash](https://www.amazon.com/dp/B0H4HMVSMR)
  -- Private credit systemic risk ([working paper](https://doi.org/10.5281/zenodo.20558733))

- [Tokens, Watts, and Geography -- AI Inference Pricing](https://doi.org/10.5281/zenodo.20543628)
  -- Four-tier framework for AI compute economics

---
*Djellal Djouad -- CrossVol Research -- 2026*
